# Import the required libraries

All the librairies in requirements.txt
- pip install -r requirements.txt

In [1]:
import requests
import json
import dotenv
import os
import time
from oauthlib.oauth2 import BackendApplicationClient
from oauthlib.oauth2 import TokenExpiredError
from requests_oauthlib import OAuth2Session
from requests.auth import HTTPBasicAuth

# Load the SSE API Key and Secret Key
- API and Secret Key generated on SSE Admin Page
- load values in ".env" file

In [2]:
token_url = os.environ.get('TOKEN_URL') or 'https://api.sse.cisco.com/auth/v2/token'

# Export/Set the environment variables
dotenv.load_dotenv()
client_id = os.environ.get('API_KEY')
client_secret = os.environ.get('API_SECRET')


# Generate the token required for authorization on API

In [3]:
class SecureAccessAPI:
    def __init__(self, url, ident, secret):
        self.url = url
        self.ident = ident
        self.secret = secret
        self.token = None

    def GetToken(self):
        auth = HTTPBasicAuth(self.ident, self.secret)
        client = BackendApplicationClient(client_id=self.ident)
        oauth = OAuth2Session(client=client)
        self.token = oauth.fetch_token(token_url=self.url, auth=auth)
        return self.token

In [4]:
for var in ['API_SECRET', 'API_KEY']:
    if os.environ.get(var) == None:
        print("Required environment variable: {} not set".format(var))
        exit()

# Get token
api = SecureAccessAPI(token_url, client_id, client_secret)
#print("Token: " + str(api.GetToken()))

token = api.GetToken()['access_token']
#print(token)

In [ ]:
url = 'https://api.sse.cisco.com/deployments/v2/regions'

In [ ]:
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {token}",
    "Accept": "application/json"
}

In [ ]:
response = requests.request('GET', url, headers=headers)



#response = requests.request('POST', url, header=headers, data = payload)

#print(response.text.encode('utf8'))
#print(json.dumps(response.json(), indent=2, ensure_ascii=False))
with open("regions.json", "w", encoding="utf-8") as f:
    json.dump(response.json(), f, indent=2, ensure_ascii=False)

# Get Private Resources

In [ ]:
headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/json"
}

url = "https://api.sse.cisco.com/policies/v2/privateResources?limit=1000"

response = requests.request('GET', url, headers=headers)
#print(response.text.encode('utf8'))
#print(json.dumps(response.json(), indent=2, ensure_ascii=False))
with open("private-resources.json", "w", encoding="utf-8") as f:
    json.dump(response.json(), f, indent=2, ensure_ascii=False)


## Create a Private Resource

In [ ]:
url = "https://api.sse.cisco.com/policies/v2/privateResources"

payload = '''{
    "name": "UdeM-Jira-3",
    "friendlyName": "UdeM-Jira-3",
    "description": "",
    "dnsServerId": 433,
    "accessTypes": [
       
        {
            "type": "client",
            "reachableAddresses": [
                "www.ecadeau-2.umontreal.ca"
            ]
        },
        { 
            "type": "network"
        }
    ],
    "resourceAddresses": [
        {
            "destinationAddr": [
                "www.ecadeau-2.umontreal.ca"
            ],
            "protocolPorts": [
                {
                    "protocol": "http/https",
                    "ports": "443,80"
                }
            ]
        }
    ],
    "resourceGroupIds": [],
    "importSource": null,
    "enforcement": "CLOUD",
    "logoUrl": ""
}'''

response = requests.request('POST', url, headers=headers, data = payload)
print(response.text.encode('utf8'))


## Variable Input from List

In [ ]:
url = "https://api.sse.cisco.com/policies/v2/privateResources"

payload = '''{
    "name": "private-resource-name",
    "friendlyName": "private-resource-friendly",
    "description": "",
    "dnsServerId": 433,
    "accessTypes": [
       
        {
            "type": "client",
            "reachableAddresses": [
                "site_rul"
            ]
        },
        { 
            "type": "network"
        }
    ],
    "resourceAddresses": [
        {
            "destinationAddr": [
                "FQDN"
            ],
            "protocolPorts": [
                {
                    "protocol": "tcp",
                    "ports": "443,80"
                }
            ]
        }
    ],
    "resourceGroupIds": [],
    "importSource": null,
    "enforcement": "CLOUD",
    "logoUrl": ""
}'''

#response = requests.request('POST', url, headers=headers, data = payload)
#print(response.text.encode('utf8'))
# Read the JSON file
with open('sample-5-private-resources.json', 'r') as f:
    data = json.load(f)

# Loop through each entry and create POST request
for entry in data:
    # Parse the existing payload
    payload_dict = json.loads(payload)
    
    # Replace the name and friendlyName with the Rule value from JSON
    payload_dict['name'] = entry['Rule']
    payload_dict['friendlyName'] = entry['Rule']
    payload_dict['accessTypes'][0]['reachableAddresses'] = [entry['Destination']]
    payload_dict['resourceAddresses'][0]['destinationAddr'] = [entry['Destination']]
    payload_dict['resourceAddresses'][0]['protocolPorts'][0]['protocol'] = entry['Protocole']
    payload_dict['resourceAddresses'][0]['protocolPorts'][0]['ports'] = entry['Port']
    
    
    
    # Convert back to JSON string
    payload_updated = json.dumps(payload_dict)
    #print(payload_updated)


    # Make the POST request
    response = requests.request('POST', url, headers=headers, data=payload_updated)
    print(f"Created resource: {entry['Rule']} - Status: {response.status_code}")

# Access Rules Section

## Access-Policy Rules Extraction

In [ ]:
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
    "Accept": "application/json"
}

url = "https://api.sse.cisco.com/policies/v2/rules?limit=1000"

response = requests.request('GET', url, headers=headers)
#print(response.text.encode('utf8'))
#print(json.dumps(response.json(), indent=2, ensure_ascii=False))
with open("access-rules.json", "w", encoding="utf-8") as f:
    json.dump(response.json(), f, indent=2, ensure_ascii=False)


In [ ]:
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
    "Accept": "application/json"
}
#url = "https://api.sse.cisco.com/policies/v2/settings?limit=1000"

url = "https://api.sse.cisco.com/policies/v2/objects/networkObjects?limit=100"


response = requests.request('GET', url, headers=headers)

print(response.text.encode('utf8'))

b'{"message":"Bad Request","code":400,"details":"error parsing limit and offset. Err: limit must be less than or equal to 100"}'
